### Imports

In [1]:
import sys
sys.dont_write_bytecode = True


import torch
import numpy as np
import random
import os

def set_seeds(seed_value=42):
    """Sets seeds for reproducibility."""
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)

set_seeds(42) 

import json
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import warnings
import logging
from datetime import datetime
warnings.filterwarnings('ignore')

from model import get_model
from config import CFG
from dataset import get_dataset_class
from transform import get_transforms
from runner import run_baseline, run_lodo

torch.manual_seed(CFG["system"]["seed"])
np.random.seed(CFG["system"]["seed"])

device = CFG["system"]["device"]
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

DS = "OfficeHome"
MODEL_NAME = "resnet18"

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu126 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1126 09:00:04.663000 43232 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Device: cuda
PyTorch: 2.8.0+cu126


### DataLoading

In [2]:
train_transform, test_transform = get_transforms(img_size=224, augment=False, use_imagenet_norm=False)

DatasetClass = get_dataset_class(DS)

ld = DatasetClass(
    data_root=CFG["datasets"][DS]["root"],
    transform=train_transform,
    batch_size=CFG["train"]["batch_size"]
)

print("\nData loaders ready!")


Data loaders ready!


### Logging

In [3]:
dataset_name = DS
base_dir = os.path.join(os.getcwd(), dataset_name)
subdirs = ["logs", "checkpoints", "plots"]

for sub in subdirs:
    os.makedirs(os.path.join(base_dir, sub), exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
log_file = os.path.join(base_dir, "logs", f"train_{timestamp}.log")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(f"{dataset_name}_logger")

logger.info(f"Initialized experiment directories for {dataset_name}")
logger.info(f"Logs: {os.path.join(base_dir, 'logs')}")
logger.info(f"Checkpoints: {os.path.join(base_dir, 'checkpoints')}")
logger.info(f"Plots: {os.path.join(base_dir, 'plots')}")

2025-11-26 09:00:06,065 | INFO | Initialized experiment directories for OfficeHome
2025-11-26 09:00:06,066 | INFO | Logs: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet34_experiments\OfficeHome\logs
2025-11-26 09:00:06,066 | INFO | Checkpoints: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet34_experiments\OfficeHome\checkpoints
2025-11-26 09:00:06,067 | INFO | Plots: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet34_experiments\OfficeHome\plots


### Setup

In [4]:
domains = CFG["datasets"][DS]["domains"]
loaders = {d: {"train": ld.get_dataloader(d, train=True), "val": ld.get_dataloader(d, train=False)} for d in domains}
ckpt_root = os.path.join(base_dir, "checkpoints")
log_dir = os.path.join(base_dir, "logs")
plots_dir = os.path.join(base_dir, "plots")
os.makedirs(ckpt_root, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)
model_factory = lambda cfg, dataset_key: get_model(cfg,dataset=DS)
optimizer_fn = lambda model: optim.AdamW(model.parameters(), lr=CFG["train"]["lr"], weight_decay=CFG["train"].get("weight_decay", 0.01))
device = CFG["system"]["device"]
epochs = CFG["train"]["epochs"]
CFG["grqo"]["num_tokens"] = 32

{
  "lodo_results": {
    "art_painting": 0.8341463414634146,
    "cartoon": 0.7974413646055437,
    "photo": 0.9580838323353293,
    "sketch": 0.6017811704834606
  },
  "timestamp": "20251004_020611"
}

### Leave One Domain Out

In [5]:
lodo_results, lodo_mean, lodo_summary = run_lodo(
    model_fn=model_factory,
    CFG=CFG,
    logger=logger,
    dataset_key=DS,
    domains=domains,
    loaders=loaders,
    optimizer_fn=optimizer_fn,
    device=device,
    ckpt_root=ckpt_root,
    log_dir=log_dir,
    epochs=20
)

2025-11-26 09:00:06,433 | INFO | === LODO: Leaving out domain 'Art' ===



=== LODO: Leaving out domain 'Art' ===


Evaluating: 100%|██████████| 19/19 [00:11<00:00,  1.72it/s]
2025-11-26 09:00:41,433 | INFO | [Art] Epoch 1/20 | Train - Loss: 2.1507, Cls: 2.1439, GRQO: 0.0068, Acc: 0.5329 | Val - Loss: 1.7468, Cls: 1.7451, GRQO: 0.0017, Acc: 0.5587
2025-11-26 09:00:41,529 | INFO | [Art] New best val acc: 0.5587


[Art] Epoch 1/20 | Train - Loss: 2.1507, Cls: 2.1439, GRQO: 0.0068, Acc: 0.5329 | Val - Loss: 1.7468, Cls: 1.7451, GRQO: 0.0017, Acc: 0.5587
[Art] New best val acc: 0.5587


Evaluating: 100%|██████████| 19/19 [00:11<00:00,  1.68it/s]
2025-11-26 09:01:16,267 | INFO | [Art] Epoch 2/20 | Train - Loss: 0.5521, Cls: 0.5493, GRQO: 0.0028, Acc: 0.8662 | Val - Loss: 1.7182, Cls: 1.7170, GRQO: 0.0012, Acc: 0.5781
2025-11-26 09:01:16,373 | INFO | [Art] New best val acc: 0.5781


...

[Real World] Epoch 18/20 | Train - Loss: 0.0127, Cls: 0.0241, GRQO: -0.0114, Acc: 0.9904 | Val - Loss: 1.1675, Cls: 1.1739, GRQO: -0.0064, Acc: 0.7537


Evaluating: 100%|██████████| 35/35 [00:47<00:00,  1.37s/it]
2025-11-26 09:57:23,753 | INFO | [Real World] Epoch 19/20 | Train - Loss: 0.0103, Cls: 0.0222, GRQO: -0.0119, Acc: 0.9910 | Val - Loss: 1.1610, Cls: 1.1668, GRQO: -0.0058, Acc: 0.7588


[Real World] Epoch 19/20 | Train - Loss: 0.0103, Cls: 0.0222, GRQO: -0.0119, Acc: 0.9910 | Val - Loss: 1.1610, Cls: 1.1668, GRQO: -0.0058, Acc: 0.7588


Evaluating: 100%|██████████| 35/35 [00:47<00:00,  1.36s/it]
2025-11-26 09:58:31,038 | INFO | [Real World] Epoch 20/20 | Train - Loss: 0.0093, Cls: 0.0218, GRQO: -0.0125, Acc: 0.9903 | Val - Loss: 1.1784, Cls: 1.1846, GRQO: -0.0061, Acc: 0.7569
2025-11-26 09:58:31,039 | INFO | [Real World] Best Acc: 0.7714
2025-11-26 09:58:31,039 | INFO | ------------------------------------------------------------
2025-11-26 09:58:31,041 | INFO | LODO finished | Mean Acc: 0.6630 | Summary saved to d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet34_experiments\OfficeHome\logs\lodo_summary_20251126_095831.json


[Real World] Epoch 20/20 | Train - Loss: 0.0093, Cls: 0.0218, GRQO: -0.0125, Acc: 0.9903 | Val - Loss: 1.1784, Cls: 1.1846, GRQO: -0.0061, Acc: 0.7569
[Real World] Best Acc: 0.7714
------------------------------------------------------------
LODO finished | Mean Acc: 0.6630
Summary saved to d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet34_experiments\OfficeHome\logs\lodo_summary_20251126_095831.json


### Baseline

In [6]:
baseline_results, baseline_mean = run_baseline(
    model_name=MODEL_NAME,
    CFG=CFG,
    logger=logger,
    dataset_key=DS,
    domains=domains,
    loaders=loaders,
    optimizer_fn=optimizer_fn,
    device=device,
    epochs=10
)

2025-11-26 09:58:31,048 | INFO | Initializing ResNet baseline: resnet18
2025-11-26 09:58:31,120 | INFO | === Baseline LODO: Leaving out domain 'Art' ===


Initializing ResNet baseline: resnet18

=== Baseline LODO: Leaving out domain 'Art' ===


2025-11-26 09:59:04,920 | INFO | [Art] Epoch 1/10 | Train - Loss: 2.2228, Acc: 0.5312 | Val Acc: 0.4887


[Art] Epoch 1/10 | Train - Loss: 2.2228, Acc: 0.5312 | Val Acc: 0.4887


2025-11-26 09:59:39,118 | INFO | [Art] Epoch 2/10 | Train - Loss: 0.8330, Acc: 0.8340 | Val Acc: 0.5270


...

[Real World] Epoch 8/10 | Train - Loss: 0.0497, Acc: 0.9886 | Val Acc: 0.7537


2025-11-26 10:25:05,081 | INFO | [Real World] Epoch 9/10 | Train - Loss: 0.0405, Acc: 0.9889 | Val Acc: 0.7505


[Real World] Epoch 9/10 | Train - Loss: 0.0405, Acc: 0.9889 | Val Acc: 0.7505


2025-11-26 10:26:08,348 | INFO | [Real World] Epoch 10/10 | Train - Loss: 0.0353, Acc: 0.9894 | Val Acc: 0.7537
2025-11-26 10:26:08,349 | INFO | [Real World] Best Val Acc: 0.7537
2025-11-26 10:26:08,349 | INFO | ------------------------------------------------------------
2025-11-26 10:26:08,350 | INFO | Baseline LODO (resnet18) finished | Mean Acc: 0.6367


[Real World] Epoch 10/10 | Train - Loss: 0.0353, Acc: 0.9894 | Val Acc: 0.7537
[Real World] Best Val Acc: 0.7537
------------------------------------------------------------
Baseline LODO (resnet18) finished | Mean Acc: 0.6367
